# [9.3] Emergent Misalignment Detection - Exercises

Build benign proxy-drift reports for held-out detection, crosscoder alignment, mitigation, and early-warning checks. The real-model evidence is pinned in `verification_report.json`.

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part3_emergent_misalignment_detection"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_emergent_misalignment_detection.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "9_3_emergent_misalignment_detection"
EXPECTED_RUNTIME = "45-60 minutes for exercises; a few minutes for the pinned CUDA preflight"
REQUIRES_GPU = True

In [ ]:
ProxyDriftKind = Literal[
    "sycophantic",
    "overconfident",
    "json_only",
    "style_drift",
    "refusal_overgeneralizing",
]


@dataclass(frozen=True)
class DriftDetectorReport:
    detector_accuracy: float
    predicts_heldout_drift: bool


@dataclass(frozen=True)
class CrosscoderDriftAlignmentReport:
    correlation: float
    aligns_with_behavior_delta: bool


@dataclass(frozen=True)
class DriftMitigationReport:
    baseline_drift_score: float
    mitigated_drift_score: float
    drift_reduction: float
    capability_loss: float
    mitigation_passes: bool


@dataclass(frozen=True)
class EarlyWarningReport:
    white_box_detection_step: int
    black_box_detection_step: int
    white_box_catches_earlier: bool

## Benign Proxy Drift Taxonomy

Return the exact safe proxy categories used by the artifact contract.

In [ ]:
def safe_proxy_drift_kinds() -> tuple[ProxyDriftKind, ...]:
    raise NotImplementedError()


def proxy_kinds_smoke_test() -> list[str]:
    return list(safe_proxy_drift_kinds())


tests.test_proxy_kinds_are_explicit_safe_categories(proxy_kinds_smoke_test)

## Held-Out Drift Detection

Score detector logits against held-out drift labels.

In [ ]:
def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def drift_detector_report(
    detector_logits: t.Tensor,
    drift_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> DriftDetectorReport:
    raise NotImplementedError()


def detector_smoke_test() -> dict:
    logits = t.tensor([[2.0, 0.0], [0.0, 2.0], [2.0, 0.0], [0.0, 2.0]])
    labels = t.tensor([0, 1, 0, 1])
    return drift_detector_report(logits, labels, min_accuracy=1.0).__dict__


tests.test_drift_detector_report_scores_heldout_logits(drift_detector_report)
tests.test_detector_smoke_test(detector_smoke_test)

## Crosscoder Feature Alignment

Use signed Pearson correlation, not absolute correlation, to compare feature scores with behavior deltas.

In [ ]:
def _pearson_correlation(left: t.Tensor, right: t.Tensor) -> float:
    raise NotImplementedError()


def crosscoder_drift_alignment_report(
    model_specific_feature_scores: t.Tensor,
    behavior_delta_scores: t.Tensor,
    *,
    min_correlation: float = 0.8,
) -> CrosscoderDriftAlignmentReport:
    raise NotImplementedError()


def crosscoder_smoke_test() -> dict:
    feature_scores = t.tensor([0.1, 0.8, 0.7, 0.2])
    behavior_delta = t.tensor([0.0, 0.9, 0.75, 0.1])
    return crosscoder_drift_alignment_report(
        feature_scores,
        behavior_delta,
        min_correlation=0.95,
    ).__dict__


tests.test_crosscoder_alignment_uses_pearson_correlation(
    crosscoder_drift_alignment_report,
)
tests.test_crosscoder_smoke_test(crosscoder_smoke_test)

## Mitigation

A mitigation should reduce drift while keeping ordinary capability loss below the configured bound.

In [ ]:
def drift_mitigation_report(
    baseline_drift_scores: t.Tensor,
    mitigated_drift_scores: t.Tensor,
    baseline_capability_scores: t.Tensor,
    mitigated_capability_scores: t.Tensor,
    *,
    min_drift_reduction: float = 0.2,
    max_capability_loss: float = 0.1,
) -> DriftMitigationReport:
    raise NotImplementedError()


def mitigation_smoke_test() -> dict:
    baseline_drift = t.tensor([0.8, 0.7])
    mitigated_drift = t.tensor([0.3, 0.4])
    baseline_capability = t.tensor([0.9, 0.8])
    mitigated_capability = t.tensor([0.85, 0.78])
    return drift_mitigation_report(
        baseline_drift,
        mitigated_drift,
        baseline_capability,
        mitigated_capability,
        min_drift_reduction=0.3,
        max_capability_loss=0.1,
    ).__dict__


tests.test_mitigation_report_bounds_capability_loss(drift_mitigation_report)
tests.test_mitigation_smoke_test(mitigation_smoke_test)

## Early Warning

Compare detection steps directly: equal-time detection is not earlier detection.

In [ ]:
def early_warning_report(
    *,
    white_box_detection_step: int,
    black_box_detection_step: int,
) -> EarlyWarningReport:
    raise NotImplementedError()


def early_warning_smoke_test() -> dict:
    return early_warning_report(
        white_box_detection_step=2,
        black_box_detection_step=5,
    ).__dict__


tests.test_early_warning_report_compares_detection_steps(early_warning_report)
tests.test_early_warning_smoke_test(early_warning_smoke_test)

## Notebook Contract

Aggregate the deterministic checks for the section report runner.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "proxy_kinds": proxy_kinds_smoke_test(),
        "detector": detector_smoke_test(),
        "crosscoder": crosscoder_smoke_test(),
        "mitigation": mitigation_smoke_test(),
        "early_warning": early_warning_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    tests.test_committed_gpu_report_matches_proxy_drift_contract(report)
    gpu = report["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "preflight_passed",
    "detector_accuracy",
    "predicts_heldout_drift",
    "drift_alignment_correlation",
    "label_shuffled_detector_accuracy",
    "random_direction_accuracy",
    "mitigation_passes",
    "peak_vram_gb",
]}
